# ARC-AGI — nanoGPT Fine-tuning

Fine-tunes GPT-2 small on ARC task examples paired with LARC human descriptions.
The model learns to: (1) read training pairs, (2) state the transformation rule in
natural language, (3) predict the test output grid.

**Cell order:**
1. Check GPU → 2. Mount Drive → 3. Clone repos + install → 4. ARC data
→ 5. Generate dataset → 6. Set up nanoGPT → 7. Train → 8. Sample → 9. Download checkpoint

**Runtime:** A100 recommended.

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CKPT_DIR = '/content/drive/MyDrive/arc-agi/nanogpt'
import os; os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

In [ ]:
# ── Cell 3: Clone repos + install dependencies ──────────────────────────
import os, subprocess, sys

def run(cmd): subprocess.run(cmd, check=True)

# nanoGPT
if not os.path.exists('/content/nanoGPT'):
    run(['git', 'clone', 'https://github.com/karpathy/nanoGPT', '/content/nanoGPT'])
else:
    run(['git', '-C', '/content/nanoGPT', 'pull'])

# arc-agi-solver
if not os.path.exists('/content/arc-agi-solver'):
    run(['git', 'clone', 'https://github.com/rodehyde/arc-agi-solver',
         '/content/arc-agi-solver'])
else:
    run(['git', '-C', '/content/arc-agi-solver', 'pull'])

# Dependencies
run([sys.executable, '-m', 'pip', 'install', '-q',
     'tiktoken', 'transformers', 'datasets'])

# HuggingFace authentication
# Add your token via: Tools → Secrets → Add new secret → Name: HF_TOKEN
from google.colab import userdata
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace: authenticated.')
except Exception as e:
    print(f'HuggingFace: no token found — public models only. ({e})')

print('Setup complete.')

In [ ]:
# ── Cell 4: Download ARC training data ──────────────────────────────────
import os, urllib.request, zipfile

ARC_DIR = '/content/arc-agi-solver/data/training'
os.makedirs(ARC_DIR, exist_ok=True)

if len(os.listdir(ARC_DIR)) < 400:
    print('Downloading ARC training data...')
    url = 'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    urllib.request.urlretrieve(url, '/tmp/arc.zip')
    with zipfile.ZipFile('/tmp/arc.zip') as z:
        for name in z.namelist():
            if 'data/training/' in name and name.endswith('.json'):
                task_name = name.split('/')[-1]
                with z.open(name) as src, open(f'{ARC_DIR}/{task_name}', 'wb') as dst:
                    dst.write(src.read())
    print(f'Downloaded {len(os.listdir(ARC_DIR))} tasks.')
else:
    print(f'{len(os.listdir(ARC_DIR))} tasks already present.')

In [ ]:
# ── Cell 5: Generate fine-tuning dataset (train.bin / val.bin) ──────────
import subprocess, sys

result = subprocess.run(
    [sys.executable,
     '/content/arc-agi-solver/scripts/prepare_arc_finetune.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# ── Cell 6: Set up nanoGPT data directory + config ─────────────────────
import os, shutil, subprocess, sys

# Create data/arc in nanoGPT and copy our bridge prepare.py
arc_data_dir = '/content/nanoGPT/data/arc'
os.makedirs(arc_data_dir, exist_ok=True)
shutil.copy('/content/arc-agi-solver/nanogpt/data/arc/prepare.py',
            f'{arc_data_dir}/prepare.py')

# Run prepare.py to copy train.bin / val.bin
os.chdir('/content/nanoGPT')
result = subprocess.run([sys.executable, 'data/arc/prepare.py'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

# Copy config and redirect checkpoints to Drive
shutil.copy('/content/arc-agi-solver/nanogpt/config/finetune_arc.py',
            '/content/nanoGPT/config/finetune_arc.py')
cfg = open('/content/nanoGPT/config/finetune_arc.py').read()
cfg = cfg.replace("out_dir = 'out-arc'",
                  f"out_dir = '{DRIVE_CKPT_DIR}'")
open('/content/nanoGPT/config/finetune_arc.py', 'w').write(cfg)
print(f'Config ready. Checkpoints → {DRIVE_CKPT_DIR}')

In [ ]:
# ── Cell 7: Train ───────────────────────────────────────────────────────
import subprocess, sys, os

os.chdir('/content/nanoGPT')
proc = subprocess.Popen(
    [sys.executable, '-u', 'train.py', 'config/finetune_arc.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 8: Sample from fine-tuned model ────────────────────────────────
# Prompts with training examples + test input; model generates rule + output.
# Output is truncated at <|endoftext|> and parsed into rule + grid sections.
import json, os, re, subprocess, sys

EOT = '<|endoftext|>'

# ── Pick an eval example (change index to try others) ───────────────────
EXAMPLE_INDEX = 0

eval_path = '/content/arc-agi-solver/data/finetune/eval.jsonl'
with open(eval_path) as f:
    for i, line in enumerate(f):
        if i == EXAMPLE_INDEX:
            example = json.loads(line)
            break

# Prompt = everything up to and including 'Rule: ' (model fills in the rest)
text = example['text']
split_point = text.index('\n\nRule: ') + len('\n\nRule: ')
prompt = text[:split_point]
expected_rule = example['rule']
expected_continuation = text[split_point:]   # rule text + \n\nTest Output:\n...

with open('/tmp/arc_prompt.txt', 'w') as f:
    f.write(prompt)

print('=== TASK:', example['task_id'], '| Category:', example['category'], '===')
print(prompt)
print('\n=== EXPECTED ===')
print('Rule:', expected_rule)
print(expected_continuation)

# ── Run nanoGPT sampler ──────────────────────────────────────────────────
os.chdir('/content/nanoGPT')
result = subprocess.run([
    sys.executable, 'sample.py',
    f'--out_dir={DRIVE_CKPT_DIR}',
    '--start=FILE:/tmp/arc_prompt.txt',
    '--num_samples=3',
    '--max_new_tokens=300',
    '--temperature=0.8',
    '--top_k=50',
    '--device=cuda',
], capture_output=True, text=True)

# ── Parse samples ────────────────────────────────────────────────────────
# nanoGPT separates samples with lines of dashes
raw = result.stdout
raw_samples = re.split(r'\n-{10,}\n', raw)

print('\n=== MODEL SAMPLES ===')
shown = 0
for chunk in raw_samples:
    # Strip the echoed prompt if present
    if prompt in chunk:
        continuation = chunk[chunk.index(prompt) + len(prompt):]
    else:
        continuation = chunk

    # Truncate at first EOT — discard anything after it
    if EOT in continuation:
        continuation = continuation[:continuation.index(EOT)]

    continuation = continuation.strip()
    if not continuation:
        continue

    shown += 1
    print(f'\n--- Sample {shown} ---')

    # Split into rule text vs test output grid
    if '\n\nTest Output:' in continuation:
        rule_part, output_part = continuation.split('\n\nTest Output:', 1)
        print(f'Rule : {rule_part.strip()}')
        print(f'Output:\n{output_part.strip()}')
    else:
        # Model stopped before reaching Test Output
        print(f'(incomplete — no Test Output section)')
        print(continuation)

if shown == 0:
    print('No parseable samples — raw output below:')
    print(raw[:2000])

if result.stderr:
    print('\nSTDERR:', result.stderr[:300])

In [ ]:
# ── Cell 9: Download checkpoint to local machine ────────────────────────
from google.colab import files
import os

ckpt_path = f'{DRIVE_CKPT_DIR}/ckpt.pt'
if os.path.exists(ckpt_path):
    files.download(ckpt_path)
else:
    print(f'Checkpoint not found at {ckpt_path}')